In [ ]:
from pathlib import Path
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

repo_root = Path.cwd()
for c in [repo_root, *repo_root.parents]:
    if (c / 'Simulation_4').exists():
        repo_root = c
        break

runner = repo_root / 'Simulation_4' / 'scripts' / 'phase9_run_validation.py'
report_path = repo_root / 'Simulation_4' / 'artifacts' / 'phase9' / 'validation_report.json'

print('Run the full validation suite from notebook:')
print(runner)
!python "{runner}"

results = json.loads(report_path.read_text(encoding='utf-8'))
print('Loaded report:', report_path)
print('Overall pass:', results.get('overall_pass'))
print('Passed/Total:', results.get('passed_checks'), '/', results.get('total_checks'))

In [ ]:
lvl1 = results.get('level1_statistical', {})
metrics = lvl1.get('summary_level1_metrics', {}).get('value', {})

outcome_df = pd.DataFrame([{
    'outcome': k,
    'rate': metrics.get(k, 0.0)
} for k in ['resolution_rate', 'escalation_rate', 'dropout_rate', 'timeout_rate']])
fig_outcomes = px.bar(outcome_df, x='outcome', y='rate', title='Level 1: Terminal Outcome Distribution (Document-Guided)')
fig_outcomes.show()

info_series = lvl1.get('check3_information_trajectory', {}).get('value', {}).get('mean_information_by_turn', {})
fr_series = lvl1.get('check4_frustration_trajectory', {}).get('value', {}).get('mean_frustration_by_turn', {})

info_df = pd.DataFrame({'turn': list(info_series.keys()), 'mean_information': list(info_series.values())}).sort_values('turn')
fr_df = pd.DataFrame({'turn': list(fr_series.keys()), 'mean_frustration': list(fr_series.values())}).sort_values('turn')

fig_info = px.line(info_df, x='turn', y='mean_information', markers=True, title='Level 1: Mean Information Trajectory')
fig_info.update_xaxes(dtick=1)
fig_info.show()

fig_fr = px.line(fr_df, x='turn', y='mean_frustration', markers=True, title='Level 1: Mean Frustration Trajectory')
fig_fr.update_xaxes(dtick=1)
fig_fr.show()

persona_tbl = lvl1.get('check5_persona_differentiation', {}).get('value', {}).get('persona_outcome_rates', {})
display(pd.DataFrame(persona_tbl).T if persona_tbl else pd.DataFrame())

In [ ]:
lvl2 = results.get('level2_transition', {})

dropout_stats = lvl2.get('check6_dropout_monotone', {}).get('value', {})
drop_df = pd.DataFrame([
    {'bucket': 'fr~0.1', 'p_dropout': dropout_stats.get('mean_pdrop_f01', 0.0)},
    {'bucket': 'fr~0.4', 'p_dropout': dropout_stats.get('mean_pdrop_f04', 0.0)},
    {'bucket': 'fr~0.8', 'p_dropout': dropout_stats.get('mean_pdrop_f08', 0.0)},
])
fig_drop = px.scatter(drop_df, x='bucket', y='p_dropout', title='Level 2: p_dropout vs Frustration Bucket')
fig_drop.show()

box_rows = []
ps = lvl2.get('check2_provide_solution_dynamics', {}).get('value', {})
ar = lvl2.get('check3_affective_repair', {}).get('value', {})
box_rows.append({'group': 'ProvideSolution failure', 'delta_frustration_mean': ps.get('mean_delta_f_failure', 0.0)})
box_rows.append({'group': 'ProvideSolution success', 'delta_frustration_mean': ps.get('mean_delta_f_success', 0.0)})
box_rows.append({'group': 'AffectiveRepair effective', 'delta_frustration_mean': ar.get('mean_delta_f_effective', 0.0)})
box_rows.append({'group': 'AffectiveRepair ineffective', 'delta_frustration_mean': ar.get('mean_delta_f_ineffective', 0.0)})
box_df = pd.DataFrame(box_rows)
fig_box = px.bar(box_df, x='group', y='delta_frustration_mean', title='Level 2: Mean delta_frustration by Action/Outcome')
fig_box.show()

diff_stats = lvl2.get('check7_difficulty_psuccess_gap', {}).get('value', {})
diff_df = pd.DataFrame([
    {'difficulty_bucket': 'low', 'mean_p_success': diff_stats.get('mean_psuccess_low_difficulty', 0.0)},
    {'difficulty_bucket': 'high', 'mean_p_success': diff_stats.get('mean_psuccess_high_difficulty', 0.0)},
])
fig_diff = px.bar(diff_df, x='difficulty_bucket', y='mean_p_success', title='Level 2: p_success at information~0.5 by Difficulty Bucket')
fig_diff.show()

In [ ]:
lvl3 = results.get('level3_rl_signal', {})
policy_metrics = lvl3.get('policy_metrics', {}).get('value', {})
pm_df = pd.DataFrame(policy_metrics).T.reset_index().rename(columns={'index': 'policy'})
if not pm_df.empty:
    pm_df = pm_df.sort_values('mean_episode_reward', ascending=False)
    fig_rewards = px.bar(pm_df, y='policy', x='mean_episode_reward', orientation='h', title='Level 3: Mean Episode Reward by Policy')
    fig_rewards.show()

    fig_scatter = px.scatter(pm_df, x='resolution_rate', y='mean_episode_reward', text='policy', title='Level 3: Resolution Rate vs Mean Reward')
    fig_scatter.update_traces(textposition='top center')
    fig_scatter.show()

display(pm_df[['policy', 'mean_episode_reward', 'std_episode_reward', 'resolution_rate', 'escalation_rate', 'mean_turns_to_resolution']])
else:
    print('No Level 3 policy metrics found in report.')

In [ ]:
lvl4 = results.get('level4_rag_coverage', {})
coverage_table = lvl4.get('check1_subflow_coverage', {}).get('value', {}).get('coverage_table', [])
retrieval_rows = lvl4.get('check5_retrieval_spot_checks', {}).get('value', {}).get('spot_checks', [])

cov_df = pd.DataFrame(coverage_table)
if not cov_df.empty:
    display(cov_df[['subflow', 'lumo_label', 'variant_id', 'passed', 'used_fallback']])
else:
    print('No coverage rows found.')

ret_df = pd.DataFrame(retrieval_rows)
if not ret_df.empty:
    display(ret_df)
else:
    print('No retrieval spot-check rows found.')

In [ ]:
rows = []
for level_key in ['level1_statistical', 'level2_transition', 'level3_rl_signal', 'level4_rag_coverage']:
    for check_name, check_result in results.get(level_key, {}).items():
        rows.append({
            'level': level_key,
            'check': check_name,
            'passed': bool(check_result.get('passed', False)),
            'value': str(check_result.get('value', ''))[:220],
            'threshold': str(check_result.get('threshold', ''))[:220],
        })

report_df = pd.DataFrame(rows)
display(report_df)

failed_df = report_df[report_df['passed'] == False]
if failed_df.empty:
    print('Recommendation: All key checks passed. Simulator is ready for RL training.')
else:
    print('Recommendation: Fix failing checks before RL training.')
    display(failed_df[['level', 'check', 'value', 'threshold']])